## Home Credit Default Risk

In [ ]:
# importing required libraries
# Other libraries are imported as per the work progressed

import pandas as pd
import numpy as np

import plotly as pt
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import make_scorer, roc_auc_score, average_precision_score, precision_score
from sklearn.preprocessing import OrdinalEncoder

from sklearn.metrics import classification_report

from xgboost import XGBClassifier
#For OS
import os


In [ ]:
Test_Data = pd.read_csv('data/application_test.csv')
Train_Data = pd.read_csv('data/application_train.csv')

In [ ]:
print(Test_Data.head())


In [ ]:
print(Train_Data.head())

In [ ]:
print("Train_Data:", Train_Data.shape)
print("Test_Data:", Test_Data.shape)

In [ ]:
print(set(Train_Data.columns) - set(Test_Data.columns))
print(set(Test_Data.columns) - set(Train_Data.columns))

In [ ]:
#Train data info
Train_Data.info()

In [ ]:
Train_Data.sample(5)

In [ ]:
Train_Data.tail()

In [ ]:
#Column understand
print(Train_Data.columns.tolist())

In [ ]:
#Columns in list
for i, col in enumerate(Train_Data.columns, start=1):
    print(i, col)

In [ ]:
print("Duplicate rows:", Train_Data.duplicated().sum())
print("Duplicate SK_ID_CURR:", Train_Data["SK_ID_CURR"].duplicated().sum())

In [ ]:
#missing values
missing = Train_Data.isna().sum().sort_values(ascending=False)

print(missing.head(20))

In [ ]:
#in percentage
missing_pct = (
    Train_Data.isna().mean() * 100
).sort_values(ascending=False)

print(missing_pct.head(20))

In [ ]:
#Target distribution
print(
    Train_Data["TARGET"]
    .value_counts(normalize=True)
    .mul(100)
)

In [ ]:
# To check the distrivution of the "Target"
Train_Data["TARGET"].value_counts().plot(kind="bar")

plt.title("Target Distribution")
plt.xlabel("TARGET")
plt.ylabel("Number of Customers")
plt.show()

In [ ]:
# To get the statistics summary of the Train_Data
Train_Data.describe().T

In [ ]:
#Relationship with Target
Train_Data.groupby("TARGET")["AMT_INCOME_TOTAL"].median()

In [ ]:
#Distribution
sns.boxplot(
    data=Train_Data,
    x="TARGET",
    y="AMT_INCOME_TOTAL"
)

plt.show()

In [ ]:
Train_Data.isnull().sum()

In [ ]:
missing = Train_Data.isnull().sum().sort_values(ascending=False)

for column, count in missing.items():
    print(f"{column}: {count}")

In [ ]:
#To check the missing values count and percentage
missing = pd.DataFrame({
    "Missing_Count": Train_Data.isnull().sum(),
    "Missing_Percentage": Train_Data.isnull().mean() * 100
})

missing = missing[missing["Missing_Count"] > 0] \
                 .sort_values("Missing_Percentage", ascending=False)

print(missing)

In [ ]:
# To check the duplicate values in Train_data
print("Duplicate rows:", Train_Data.duplicated().sum())
print("Duplicate SK_ID_CURR:", Train_Data["SK_ID_CURR"].duplicated().sum())

In [ ]:
#To check the duplicate values in Test data
print("Test duplicate rows:", Test_Data.duplicated().sum())
print("Test duplicate SK_ID_CURR:", Test_Data["SK_ID_CURR"].duplicated().sum())

In [ ]:
#To check the data types of Train data
print(Train_Data.dtypes.value_counts())

In [ ]:
# To check the numeric and categorical data types
numeric_cols_dtype = Train_Data.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_cols_dtype = Train_Data.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical columns:", len(numeric_cols_dtype))
print("Categorical columns:", len(categorical_cols_dtype))

In [ ]:
# To check the summary of the AMT_INCOME_TOTAL 
Train_Data["AMT_INCOME_TOTAL"].describe()

In [ ]:
# To check the summary of the NAME_CONTRACT_TYPE
Train_Data["NAME_CONTRACT_TYPE"].value_counts()

In [ ]:
# To check the summary of the OCCUPATION_TYPE
Train_Data["OCCUPATION_TYPE"].value_counts(dropna=False)

In [ ]:

# Calculate percentages of different categories of the OCCUPATION_TYPE
occupation_pct = (
    Train_Data["OCCUPATION_TYPE"]
    .fillna("Missing")
    .value_counts(normalize=True)
    .mul(100)
    .sort_values()
)

# Plot
fig, ax = plt.subplots(figsize=(10, 7))

bars = ax.barh(
    occupation_pct.index,
    occupation_pct.values,
    color="navy",
    edgecolor="black",
    linewidth=0.8
)

# Add percentage labels
for bar, value in zip(bars, occupation_pct.values):
    ax.text(
        value + 0.2,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.1f}%",
        va="center",
        fontsize=10
    )

ax.set_xlabel("Percentage of Applicants (%)", fontsize=12)
ax.set_ylabel("Occupation Type", fontsize=12)
ax.set_title(
    "Applicant Distribution by Occupation Type",
    fontsize=15,
    fontweight="bold"
)

ax.set_xlim(0, occupation_pct.max() + 5)
ax.grid(axis="x", alpha=0.25)

# Keep a clean border
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1)

plt.tight_layout()
plt.show()

#checking for summary
#numeric variable eda

In [ ]:
Train_Data["AMT_INCOME_TOTAL"].describe()

In [ ]:
Train_Data["AMT_CREDIT"].describe()

In [ ]:
Train_Data["AMT_ANNUITY"].describe()

In [ ]:
Train_Data["AMT_GOODS_PRICE"].describe()

In [ ]:
Train_Data["DAYS_BIRTH"].describe()

In [ ]:
(Train_Data['DAYS_BIRTH']/-365).describe()

In [ ]:

# Convert DAYS_BIRTH to age in years
age = Train_Data["DAYS_BIRTH"] / -365

plt.figure(figsize=(12, 7))

sns.histplot(
    age,
    bins=30,
    kde=True,
    edgecolor="white",
    linewidth=0.8
)

plt.xlabel("Age of Client (Years)", fontsize=12)
plt.ylabel("Number of Clients", fontsize=12)
plt.title(
    "Distribution of Client Age",
    fontsize=16,
    fontweight="bold",
    pad=15
)

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.3
)

plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

plt.tight_layout()
plt.show()

Interpretation: People in the age range 30-40 years are the most applicants. Which seems pretty normal.

In [ ]:
Train_Data["DAYS_EMPLOYED"].describe()

In [ ]:
(Train_Data['DAYS_EMPLOYED']/365).describe()

In [ ]:
# Check an important data-quality issue in DAYS_EMPLOYED

sentinel_value = 365243

sentinel_count = (
    Train_Data["DAYS_EMPLOYED"] == sentinel_value
).sum()

sentinel_percent = (
    Train_Data["DAYS_EMPLOYED"] == sentinel_value
).mean() * 100

print(
    f"DAYS_EMPLOYED == {sentinel_value}: "
    f"{sentinel_count:,} rows ({sentinel_percent:.2f}%)"
)

In [ ]:
# Graph of the valid employment and impossible duration of ~ 1001 years
sentinel_value = 365243

sentinel_value_pct = (
    Train_Data["DAYS_EMPLOYED"].eq(sentinel_value).mean() * 100
)

valid_pct = 100 - sentinel_value_pct

categories = [
    "Valid employment duration",
    "Sentinel value \n(~1,001 years)"
]

percentages = [
    valid_pct,
    sentinel_value_pct
]

# Create graph
fig, ax = plt.subplots(figsize=(10, 7))

bars = ax.bar(
    categories,
    percentages,
    width=0.55,
    color = ["#17365D", "black"]
)

# Add percentage labels
for bar, value in zip(bars, percentages):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1.5,
        f"{value:.1f}%",
        ha="center",
        va="bottom",
        fontsize=15,
        fontweight="bold"
    )

# Title
ax.set_title(
    "Employment Duration Data Quality(~10001 years employment)",
    fontsize=18,
    fontweight="bold",
    pad=15
)

# Axis labels
ax.set_xlabel(
    "Employment Duration Category",
    fontsize=12
)

ax.set_ylabel(
    "Percentage of Applicants (%)",
    fontsize=12
)

# Correct axis limits
ax.set_ylim(0, 100)

# Y-axis ticks
ax.set_yticks(range(0, 101, 10))

# Grid behind bars
ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

ax.set_axisbelow(True)

# Keep a complete border
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.2)

# Annotation
ax.text(
    0.5,
    0.93,
    "18% of applicants had an impossible\n"
    "employment duration (~1,000 years)",
    transform=ax.transAxes,
    ha="center",
    va="center",
    fontsize=10,
    fontweight="bold",
    bbox=dict(
        boxstyle="round,pad=0.5",
        facecolor="white",
        edgecolor="black",
        linewidth=1.2,
        alpha=0.95
    )
)

plt.tight_layout()
plt.show()

In [ ]:
Train_Data["CNT_CHILDREN"].describe()

In [ ]:
Train_Data["CNT_FAM_MEMBERS"].describe()

#Categorical variables eda

In [ ]:
categorical_cols_dtype = Train_Data.select_dtypes(
    include=["object", "category"]
).columns

for col in categorical_cols_dtype:
    print(col)


In [ ]:

for col in categorical_cols_dtype:
    print(f"\n--- {col} ---")
    print(Train_Data[col].value_counts(dropna=False))

In [ ]:
Train_Data["NAME_CONTRACT_TYPE"].value_counts()

In [ ]:
corr = Train_Data[numeric_cols_dtype].corr()

In [ ]:
target_corr = (
    corr["TARGET"]
    .drop("TARGET")
    .sort_values()
)

print(target_corr)

### Preprocessing


In [ ]:
#Keep ID separately
Train_ID = Train_Data["SK_ID_CURR"]

In [ ]:
#Target
y = Train_Data["TARGET"]

In [ ]:
#Features
X = Train_Data.drop(columns=["SK_ID_CURR", "TARGET"])

In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
#Split into training and validation
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [ ]:
# Checked for the shape of the splitted data
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

In [ ]:
def add_engineered_features(data):
    """Create interpretable features from application-level variables."""
    data = data.copy()

    # DAYS_EMPLOYED uses a sentinel value for a special case.
    data["DAYS_EMPLOYED"] = data["DAYS_EMPLOYED"].replace(
        sentinel_value,
        np.nan,
    )

    # Convert relative day counts into more interpretable years.
    data["AGE_YEARS"] = -data["DAYS_BIRTH"] / 365
    data["EMPLOYED_YEARS"] = (
        -data["DAYS_EMPLOYED"] / 365
    )

    # Financial ratios.
    #data["CREDIT_INCOME_RATIO"] = (
    #    data["AMT_CREDIT"]
    #    / data["AMT_INCOME_TOTAL"].replace(0, np.nan)
   # )

    #data["ANNUITY_INCOME_RATIO"] = (
    #    data["AMT_ANNUITY"]
    #    / data["AMT_INCOME_TOTAL"].replace(0, np.nan)
    #)

   # data["CREDIT_TERM_YEARS"] = (
   #     data["AMT_CREDIT"]
   #     / data["AMT_ANNUITY"].replace(0, np.nan)
    #)



    return data


X_train_fe = add_engineered_features(X_train)
X_val_fe = add_engineered_features(X_val)

print("Original feature count:", X_train.shape[1])
print("Feature count after engineering:", X_train_fe.shape[1])

new_features = sorted(
    set(X_train_fe.columns) - set(X_train.columns)
)

print("\nNew features:")
print(new_features)


In [ ]:
# --- Visualize relationships: does age relate to default risk? ---
train_eda = X_train_fe.copy()
train_eda['TARGET'] = y_train

fig, ax = plt.subplots(figsize=(7,4))
sns.kdeplot(data=train_eda, x='AGE_YEARS', hue='TARGET', common_norm=False, ax=ax)
ax.set_title('Age distribution by repayment outcome')
ax.set_xlabel('Client age (years)')
plt.tight_layout()
plt.show()


In [ ]:
# --- Correlation analysis (numeric features vs. TARGET) ---
numeric_train = Train_Data.select_dtypes(include=np.number)

corr_with_target = (
    numeric_train.corr()['TARGET']
    .drop('TARGET')
    .sort_values()
)

fig, ax = plt.subplots(figsize=(7, 6))

# Top 5 negative + top 5 positive correlations
top_corr = pd.concat([
    corr_with_target.head(5),
    corr_with_target.tail(5)
])

colors = ['#C44E52' if v < 0 else '#4C72B0' for v in top_corr.values]

top_corr.plot(
    kind='barh',
    ax=ax,
    color=colors
)

ax.set_title('Strongest Linear Correlations with TARGET')
ax.set_xlabel('Correlation Coefficient')

plt.tight_layout()
plt.show()


In [ ]:
## Grouped as Categorical and numerical variables
cat_cols = X_train_fe.select_dtypes(
    include=["object"]
).columns.tolist() + ["CNT_CHILDREN","CNT_FAM_MEMBERS", "REGION_RATING_CLIENT","REGION_RATING_CLIENT_W_CITY"]

print(cat_cols)

#ord_cols = ["AMT_REQ_CREDIT_BUREAU_HOUR","AMT_REQ_CREDIT_BUREAU_DAY","AMT_REQ_CREDIT_BUREAU_WEEK","AMT_REQ_CREDIT_BUREAU_MON","AMT_REQ_CREDIT_BUREAU_QRT","AMT_REQ_CREDIT_BUREAU_YEAR"]

binary_cols = ["FLAG_MOBIL","FLAG_EMP_PHONE","FLAG_WORK_PHONE","FLAG_CONT_MOBILE","FLAG_PHONE","FLAG_EMAIL","REG_REGION_NOT_LIVE_REGION","REG_REGION_NOT_WORK_REGION","LIVE_REGION_NOT_WORK_REGION",
"REG_CITY_NOT_LIVE_CITY",
"REG_CITY_NOT_WORK_CITY",
"LIVE_CITY_NOT_WORK_CITY","FLAG_DOCUMENT_2","FLAG_DOCUMENT_3",
"FLAG_DOCUMENT_4","FLAG_DOCUMENT_5","FLAG_DOCUMENT_6",
"FLAG_DOCUMENT_7","FLAG_DOCUMENT_8","FLAG_DOCUMENT_9","FLAG_DOCUMENT_10","FLAG_DOCUMENT_11","FLAG_DOCUMENT_12","FLAG_DOCUMENT_13",
"FLAG_DOCUMENT_14","FLAG_DOCUMENT_15","FLAG_DOCUMENT_16","FLAG_DOCUMENT_17",
"FLAG_DOCUMENT_18","FLAG_DOCUMENT_19","FLAG_DOCUMENT_20","FLAG_DOCUMENT_21"]

numeric_cols =X_train_fe.drop(columns=cat_cols+binary_cols).columns.tolist()
print(numeric_cols)
print(len(numeric_cols))


print(len(cat_cols)+len(numeric_cols)+len(binary_cols))

In [ ]:
#proportion of defaults is similar in train and validation.
print("Training target:")
print(y_train.value_counts(normalize=True))

print("\nValidation target:")
print(y_val.value_counts(normalize=True))

## Preprocessing pipeline

In [ ]:
#Create preprocessing pipelines
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("scaler", StandardScaler())
])

#Categorical features
#Fill missing values with most frequent category
#One-hot encode

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        drop="first"
    ))
])

binary_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            drop="first"
        ))
])

#ordinal_pipeline = Pipeline([
#    ("imputer", SimpleImputer(strategy="most_frequent")),
#    ("encoder", OrdinalEncoder(
#        handle_unknown="ignore"
#    ))
#])


#Combine them

preprocessor = ColumnTransformer([
    ("num_cols", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, cat_cols),
    ("binary", binary_pipeline, binary_cols),
    #("ordinal", ordinal_pipeline, ord_cols)
])

In [ ]:
print("Overlap:", set(cat_cols) & set(binary_cols))

### Modeling

#Dummy Classifier

In [ ]:
from sklearn.dummy import DummyClassifier

dummy_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent"))
])

dummy_model.fit(X_train_fe, y_train)

In [ ]:
y_pred_dummy = dummy_model.predict(X_val_fe)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Accuracy:", accuracy_score(y_val, y_pred_dummy))
print("Precision:", precision_score(y_val, y_pred_dummy, zero_division=0))
print("Recall:", recall_score(y_val, y_pred_dummy, zero_division=0))
print("F1:", f1_score(y_val, y_pred_dummy, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_dummy))

print("\nClassification Report:")
print(classification_report(y_val, y_pred_dummy, zero_division=0))

## Logistic Regression

Since it is highly imbalanced data, considered class_weight = balanced for all the model.

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

lr_model = Pipeline([
    ("preprocessor", preprocessor),
    ("lr_model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

In [ ]:
# To fit model 
lr_model.fit(X_train_fe, y_train)

In [ ]:
## Predict validation results
y_val_pred = lr_model.predict(X_val_fe)

In [ ]:
# Predict default probability
y_val_prob = lr_model.predict_proba(X_val_fe)[:, 1]

In [ ]:
## Import evaluation metrics
# Calculate ROC-AUC and PR-AUC
# Show confusion matrix
# Show classification report
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
# Calculate ROC-AUC and PR-AUC
print("ROC-AUC:",
      roc_auc_score(y_val, y_val_prob))

print("PR-AUC:",
      average_precision_score(y_val, y_val_prob))

# Show confusion matrix
cm = confusion_matrix(y_val, y_val_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Repay", "Default"],
    yticklabels=["Repay", "Default"]
)

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred))

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from scipy.stats import loguniform
import time

# Use the SAME 5-fold CV strategy for both searches so results are comparable.
# Cross-validation happens ONLY on X_train_fe/y_train -> the test set stays untouched.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ---------------------------------------------------------------
# 1) RANDOMIZED SEARCH — broad, cheap exploration of the parameter space
# ---------------------------------------------------------------
# C = inverse regularization strength (small C -> stronger regularization).
# loguniform(1e-4, 1e2) samples C on a log scale, which is the right way to
# search C since its effect is roughly log-linear.
param_dist = {
    'lr_model__C': loguniform(1e-3, 1e2),
    'lr_model__class_weight': [None, 'balanced'],
}

random_search = RandomizedSearchCV(
    estimator=lr_model,
    param_distributions=param_dist,
    n_iter=20,                 # 20 random combos x 5 folds = 100 fits
    scoring='roc_auc',
    cv=cv,
    random_state=42,
    n_jobs=1,
    #verbose=2,
)

t0 = time.time()
random_search.fit(X_train_fe, y_train)
print("RandomizedSearchCV time:", round(time.time() - t0, 1), "sec")
print("Best CV ROC-AUC:", round(random_search.best_score_, 4))
print("Best params found:", random_search.best_params_)

In [ ]:

# ---------------------------------------------------------------
# 2) GRID SEARCH — narrow, exhaustive search AROUND what random search found
# ---------------------------------------------------------------
best_C = random_search.best_params_["lr_model__C"]
best_cw = random_search.best_params_["lr_model__class_weight"]

# Build a tight grid centered on best_C (half to double, on a log scale)
C_grid = [best_C * f for f in (0.25, 0.5, 1, 2, 4)]

param_grid = {
    'lr_model__C': C_grid,
    'lr_model__class_weight': [best_cw],   # keep the class_weight random search preferred
}

grid_search = GridSearchCV(
    estimator=lr_model,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=2,
)

t0 = time.time()
grid_search.fit(X_train_fe, y_train)
print("GridSearchCV time:", round(time.time() - t0, 1), "sec")
print("Best CV ROC-AUC:", round(grid_search.best_score_, 4))
print("Best params:", grid_search.best_params_)



In [ ]:
# ---------------------------------------------------------------
# 3) FINAL COMPARISON on the held-out test set (only look at this once!)
# ---------------------------------------------------------------
default_model = lr_model
default_model.fit(X_train_fe, y_train)
default_auc = roc_auc_score(y_val, default_model.predict_proba(X_val_fe)[:, 1])

tuned_model = grid_search.best_estimator_   # already refit on full X_train_fe by GridSearchCV
tuned_auc = roc_auc_score(y_val, tuned_model.predict_proba(X_val_fe)[:, 1])

print(f"Default LogisticRegression test ROC-AUC: {default_auc:.4f}")
print(f"Tuned   LogisticRegression test ROC-AUC: {tuned_auc:.4f}")
print(f"Improvement: {tuned_auc - default_auc:+.4f}")

### Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier


dt_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ))
])

In [ ]:
dt_pipeline.fit(X_train_fe, y_train)

In [ ]:
y_pred = dt_pipeline.predict(X_val_fe)
y_prob = dt_pipeline.predict_proba(X_val_fe)[:, 1]

##Evaluate ROC-AUC : Baseline decision tree performance

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)

print("Accuracy:", accuracy_score(y_val, y_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_prob))
print("Recall:", recall_score(y_val, y_pred))
print("Precision:", precision_score(y_val, y_pred))
print("F1:", f1_score(y_val, y_pred))
print("F2:", fbeta_score(y_val, y_pred, beta=2))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_pred))
print("classification report:\n", classification_report(y_val, y_pred))

Calculating training and validation probabilities
Here tree is overfitting and we are not restricting any parameters

In [ ]:
from sklearn.metrics import roc_auc_score

# Training probabilities
y_train_prob = dt_pipeline.predict_proba(X_train_fe)[:, 1]
#print("y_train_prob", y_train_prob)

# Validation probabilities
y_val_prob = dt_pipeline.predict_proba(X_val_fe)[:, 1]

#rint("y_val_prob", y_val_prob)

train_roc_auc = roc_auc_score(y_train, y_train_prob)
val_roc_auc = roc_auc_score(y_val, y_val_prob)

print("Train ROC-AUC:", train_roc_auc)
print("Validation ROC-AUC:", val_roc_auc)
print("ROC-AUC Gap:", train_roc_auc - val_roc_auc)

Decision Tree: ##Hyperparameter tuning

Note: RandomizedSearchCV was used to efficiently tune hyperparameters while reducing computational time.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

param_dist = {
    "model__criterion": ["gini", "entropy"],
    "model__max_depth": [5, 8, 10, 12, 15, 20],
    "model__min_samples_split": [50, 100, 200, 500],
    "model__min_samples_leaf": [20, 50, 100, 200, 500],
    "model__max_features": [None, "sqrt", "log2"]
}

In [ ]:
#Stratified random search
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

dt_search = RandomizedSearchCV(
    estimator=dt_pipeline,
    param_distributions=param_dist,
    n_iter=20,
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)

In [ ]:
dt_search.fit(X_train_fe, y_train)

In [ ]:
print("Best parameters:")
print(dt_search.best_params_)

print("Best CV ROC-AUC:")
print(dt_search.best_score_)

In [ ]:
best_dt = dt_search.best_estimator_

y_val_prob_dt = best_dt.predict_proba(X_val_fe)[:, 1]
y_val_pred_dt = best_dt.predict(X_val_fe)


Result: For Decision Tree Hyperparameter Tuning

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)

print("Accuracy:", accuracy_score(y_val, y_val_pred_dt))
print("ROC-AUC:", roc_auc_score(y_val, y_val_prob_dt))
print("PR-AUC:", average_precision_score(y_val, y_val_prob_dt))
print("Recall:", recall_score(y_val, y_val_pred_dt))
print("Precision:", precision_score(y_val, y_val_pred_dt))
print("F1:", f1_score(y_val,y_val_pred_dt))
print("F2:", fbeta_score(y_val, y_val_pred_dt, beta=2))
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_dt))
print("classification report:\n", classification_report(y_val, y_val_pred_dt))


In [ ]:
print(dt_search.best_params_)

#Checking for overfitting

In [ ]:
#Get predicted probabilities for the training data
y_train_prob_dt = best_dt.predict_proba(X_train_fe)[:, 1]

#Calculate ROC-AUC on training data
train_auc_dt = roc_auc_score(y_train, y_train_prob_dt)
#Calculate ROC-AUC on validation data
val_auc_dt = roc_auc_score(y_val, y_val_prob_dt)

print("Train ROC-AUC:", train_auc_dt)
print("Validation ROC-AUC:", val_auc_dt)
print("ROC-AUC Gap:", train_auc_dt - val_auc_dt)

Interpretation: Here we found that, there is improvement in the Decision Tree model from baseline decision tree model to hyperparameter tuned. model. So, Hyperparameter tuning can improvise the capacity of classification of unseen data.

The baseline Decision Tree severely overfit the training data. After hyperparameter tuning, validation ROC-AUC improved from 0.541 to 0.722, while the ROC-AUC gap decreased from 0.459 to 0.030. The tuned model achieved 67.1% recall, but precision remained low at 14.9%.

- CV ROC-AUC: 0.7143
- Validation ROC-AUC: 0.7223
- PR-AUC: 0.2027
- Recall: 0.6711
- Precision: 0.1493
- F1: 0.2442
- F2: 0.3950
- Train–validation ROC-AUC gap: 0.0296

For Business:The tuned Decision Tree substantially reduced overfitting and can identify around 67% of actual defaulters, making it useful for risk screening, although its low precision means additional assessment is needed before making lending decisions.

**************************************************************End of Decision Tree **************************************************

###Random Forest

#Random forest pipeline

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

In [ ]:
rf_pipeline.fit(X_train_fe, y_train)

In [ ]:
y_val_pred_rf = rf_pipeline.predict(X_val_fe)
y_val_prob_rf = rf_pipeline.predict_proba(X_val_fe)[:, 1]

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    recall_score,
    precision_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)

print("Accuracy:", accuracy_score(y_val, y_val_pred_rf))
print("ROC-AUC:", roc_auc_score(y_val, y_val_prob_rf))
print("PR-AUC:", average_precision_score(y_val, y_val_prob_rf))
print("Recall:", recall_score(y_val, y_val_pred_rf))
print("Precision:", precision_score(y_val, y_val_pred_rf))
print("F1:", f1_score(y_val, y_val_pred_rf))
print("F2:", fbeta_score(y_val, y_val_pred_rf, beta=2))
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_rf))
print("classification report:\n", classification_report(y_val, y_val_pred_rf))


In [ ]:
#Check for overfiting
y_train_prob_rf = rf_pipeline.predict_proba(X_train_fe)[:, 1]

train_auc_rf = roc_auc_score(y_train, y_train_prob_rf)
val_auc_rf = roc_auc_score(y_val, y_val_prob_rf)

print("Train ROC-AUC:", train_auc_rf)
print("Validation ROC-AUC:", val_auc_rf)
print("ROC-AUC Gap:", train_auc_rf - val_auc_rf)

#Random Forest : Hyperparameter Tuning

In [ ]:
#param_dist_rf = {
 #   "model__n_estimators": [100, 150],
  #  "model__max_depth": [8, 12, 15],
   # "model__min_samples_split": [100, 200, 500],
    #"model__min_samples_leaf": [50, 100, 200],
    #"model__max_features": ["sqrt", "log2"],
    #"model__criterion": ["gini"]
#}

param_dist_rf = {
    "model__n_estimators": [50, 100],      # ✅ not 200-500
    "model__max_depth": [3, 5, 10],        # ✅ limit depth
    "model__max_features": ["sqrt"],       # ✅ fix one value
}

In [ ]:
cv = StratifiedKFold(
    n_splits=3,       
    shuffle=True,
    random_state=42
)

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_dist_rf,
    n_iter=20,            
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=True,            
    error_score=0         
)

In [ ]:
rf_search.fit(X_train_fe, y_train)

In [ ]:
print("Best parameters:")
print(rf_search.best_params_)

print("Best CV ROC-AUC:")
print(rf_search.best_score_)

In [ ]:
best_rf = rf_search.best_estimator_

y_pred_rf = best_rf.predict(X_val_fe)
y_prob_rf = best_rf.predict_proba(X_val_fe)[:, 1]

Result: Hyperparameter tuning
Random Forest

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)
print("Accuracy:", accuracy_score(y_val,y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_val, y_prob_rf))
print("Recall:", recall_score(y_val, y_pred_rf))
print("Precision:", precision_score(y_val, y_pred_rf))
print("F1:", f1_score(y_val, y_pred_rf))
print("F2:", fbeta_score(y_val, y_pred_rf, beta=2))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_rf))
print("classification report:\n", classification_report(y_val, y_pred_rf))

#Feature importance: random forest

In [ ]:
best_rf = rf_random.best_estimator_

rf_model = best_rf.named_steps["model"]
preprocessor_fitted = best_rf.named_steps["preprocessor"]

feature_names = preprocessor_fitted.get_feature_names_out()

importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

importance.head(20)

In [ ]:
## Feature importance
import matplotlib.pyplot as plt

top_15 = importance.head(15).sort_values("Importance")

plt.figure(figsize=(10, 7))
plt.barh(top_15["Feature"], top_15["Importance"])
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Top 15 Features – Random Forest")
plt.tight_layout()
plt.show()

************************************************************** End of Random Forest *****************************************

##XGBoost: Baseline 

In [ ]:
from xgboost import XGBClassifier

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric="auc",
        random_state=42,
        n_jobs=-1
    ))
])

In [ ]:
xgb_pipeline.fit(X_train_fe, y_train)

In [ ]:
y_pred_xgb = xgb_pipeline.predict(X_val_fe)

y_prob_xgb = xgb_pipeline.predict_proba(X_val_fe)[:, 1]

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)

print("Accuracy:", accuracy_score(y_val, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_val, y_prob_xgb))
print("Recall:", recall_score(y_val, y_pred_xgb))
print("Precision:", precision_score(y_val, y_pred_xgb))
print("F1:", f1_score(y_val, y_pred_xgb))
print("F2:", fbeta_score(y_val, y_pred_xgb, beta=2))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_xgb))
print("classification report:\n", classification_report(y_val, y_pred_xgb))

Feature Importance: XGBoost

In [ ]:
### Get feature importance
xgb_pipeline.fit(X_train_fe, y_train)

# Get the fitted preprocessor
preprocessor_fitted = xgb_pipeline.named_steps["preprocessor"]

# Get feature names after preprocessing
feature_names = preprocessor_fitted.get_feature_names_out()

# Get XGBoost feature importance
importance = xgb_pipeline.named_steps["model"].feature_importances_


### Create a DataFrame
feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
})

feature_importance_df = feature_importance_df.sort_values(
    "Importance",
    ascending=False
)


### Plot Top 15 Features

top_features = feature_importance_df.head(15).sort_values(
    "Importance",
    ascending=False
)

plt.figure(figsize=(12, 8))

sns.barplot(
    data=top_features,
    x="Importance",
    y="Feature"
)

plt.title(
    "Top 15 Features Influencing Loan Default Prediction",
    fontsize=16,
    fontweight="bold",
    pad=15
)

plt.xlabel("Feature Importance", fontsize=12)
plt.ylabel("Features", fontsize=12)

plt.grid(
    axis="x",
    linestyle="--",
    alpha=0.3
)

plt.tight_layout()
plt.show()

In [ ]:
# Test on new applications
##Test Data

In [ ]:
print(Test_Data.shape)
print(Test_Data.head())
print(Test_Data.columns.tolist())

In [ ]:
print("TARGET" in Test_Data.columns)


#### Feature Engineering for Test_Data

In [ ]:
Test_Data_fe = add_engineered_features(Test_Data)

print(Test_Data.shape)

print([
    col for col in Test_Data.columns
    if col in [
        "AGE_YEARS",
        "EMPLOYED_YEARS",
        #"CREDIT_INCOME_RATIO",
        #"ANNUITY_INCOME_RATIO",
        #"CREDIT_TERM_YEARS"
    ]
])

## Confirming one more time for the shapes of the data

In [ ]:
train_features = set(X_train_fe.columns) #- {"SK_ID_CURR"}
test_features = set(Test_Data_fe.columns) - {"SK_ID_CURR"}

print("Same features:", train_features == test_features)

In [ ]:
print("Only in Train:")
print(train_features - test_features)

print("\nOnly in Test:")
print(test_features - train_features)

## Checking once again for columns in both data

In [ ]:
print(X_train_fe.columns.equals(Test_Data_fe.columns))

In [ ]:
print(
    set(X_train_fe.columns) == set(Test_Data_fe.columns)
)

In [ ]:
Test_Data_fe = Test_Data_fe[X_train_fe.columns]

In [ ]:
print(X_train_fe.columns.equals(Test_Data_fe.columns))

In [ ]:
# Comparing both columns
comparison = pd.DataFrame({
    "Train Columns": X_train_fe.columns,
    "Test Columns": Test_Data_fe.columns
})

comparison["Same"] = (
    comparison["Train Columns"] == comparison["Test Columns"]
)

print(comparison.to_string(index=False))

print("Train shape:", X_train_fe.shape)
print("Test shape:", Test_Data_fe.shape)

## XGB for test data

In [ ]:
xgb_pipeline

In [ ]:
test_prob = xgb_pipeline.predict_proba(Test_Data_fe)[:, 1]

In [ ]:
# print("IDs:", len(test_ids))
# print("Probabilities:", len(test_prob))

In [ ]:
#Jinja2 helps Python create dynamic formatted content, such as HTML pages or tables.
pip install jinja2

In [ ]:
# Risk calculated for each SK_ID_CURR of test data and considered 0.42 as threshold value.
test_ids = Test_Data["SK_ID_CURR"]
best_threshold = 0.42

# Create predictions DataFrame
test_predictions = pd.DataFrame({
    "SK_ID_CURR": test_ids.values,
    "Predicted_Probability": test_prob
})

# Business decision
test_predictions["Risk_Decision"] = np.where(
    test_predictions["Predicted_Probability"] >= best_threshold,
    "Flag for Review",
    "Lower Risk"
)

# Binary prediction
test_predictions["Predicted_Default"] = (
    test_predictions["Predicted_Probability"] >= best_threshold
).astype(int)

test_predictions.head(5).style.set_properties(
    **{'text-align': 'center'}
).set_table_styles([
    {
        'selector': 'th',
        'props': [('text-align', 'center')]
    }
])

In [ ]:
# print("Predicted defaults:", test_pred.sum())
# print("Predicted non-defaults:", (test_pred == 0).sum())
# total = len(test_pred)

# print("Predicted defaults:", test_pred.sum())
# print("Predicted default rate:", test_pred.mean() * 100, "%")

In [ ]:
#Predicted Default Risk Distribution for Test Data
plt.figure(figsize=(10, 6))

sns.histplot(
    test_prob,
    bins=30,
    kde=True
)

plt.xlabel("Predicted Probability of Default")
plt.ylabel("Number of Applicants")
plt.title("Predicted Default Risk Distribution – Application Test Data")

plt.tight_layout()
plt.show()

***************************************************************************************************************************